In [8]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# Load API keys from .env file
load_dotenv(override=True)

# Initialize ChatGroq LLM using an ACTIVE model from your list
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)

In [9]:
# Create a stateless agent 
stateless_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a helpful assistant."
)

print("=" * 70)
print("TASK 1: STATELESS AGENT (WATCH IT FORGET)")
print("=" * 70)

# Tell the agent a personal detail
res1 = stateless_agent.invoke({
    "messages": [HumanMessage(content="My favorite color is green.")]
})
print("Run 1 (Tell):", res1["messages"][-1].content)

#  Ask the agent to recall the detail
res2 = stateless_agent.invoke({
    "messages": [HumanMessage(content="What is my favorite color?")]
})
print("\nRun 2 (Recall):", res2["messages"][-1].content)

TASK 1: STATELESS AGENT (WATCH IT FORGET)
Run 1 (Tell): That's a great choice! Green is often associated with nature, growth, and harmony. It's a very calming and refreshing color.

Do you have a specific shade of green that you prefer, like emerald, forest, or mint?

Run 2 (Recall): I don't know your favorite color! As an AI, I don't have access to your personal information or preferences unless you've shared them with me in this conversation.

If you'd like, you can tell me your favorite color, and I'll remember it for the rest of our chat! 😊


In [10]:
#  Instantiate an in-memory checkpointer
checkpointer = MemorySaver()

# Create an agent equipped with memory persistence
memory_agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
    system_prompt="You are a helpful assistant."
)

# Define a specific thread configuration
config_thread_1 = {"configurable": {"thread_id": "session_alpha"}}

print("=" * 70)
print("TASK 2: AGENT WITH CHECKPOINTER (THREAD 1)")
print("=" * 70)

# Tell the agent the detail 
res_mem1 = memory_agent.invoke(
    {"messages": [HumanMessage(content="My favorite color is green.")]},
    config=config_thread_1
)
print("Run 1 (Tell):", res_mem1["messages"][-1].content)

# Ask the agent 
res_mem2 = memory_agent.invoke(
    {"messages": [HumanMessage(content="What is my favorite color?")]},
    config=config_thread_1
)
print("\nRun 2 (Recall):", res_mem2["messages"][-1].content)

print("\n" + "-" * 40)
print("INSPECTING STATE MESSAGE HISTORY LIST")
print("-" * 40)
for i, msg in enumerate(res_mem2["messages"]):
    print(f"{i + 1}. [{msg.type.upper()}]: {msg.content}")

TASK 2: AGENT WITH CHECKPOINTER (THREAD 1)
Run 1 (Tell): That's a great choice! Green is often associated with nature, growth, and harmony. It's a very calming and refreshing color.

Do you have a specific shade of green that you prefer, like emerald, forest, or mint?

Run 2 (Recall): Your favorite color is green.

----------------------------------------
INSPECTING STATE MESSAGE HISTORY LIST
----------------------------------------
1. [HUMAN]: My favorite color is green.
2. [AI]: That's a great choice! Green is often associated with nature, growth, and harmony. It's a very calming and refreshing color.

Do you have a specific shade of green that you prefer, like emerald, forest, or mint?
3. [HUMAN]: What is my favorite color?
4. [AI]: Your favorite color is green.


In [12]:
# Define a new thread ID 
config_thread_2 = {"configurable": {"thread_id": "session_beta"}}

print("=" * 70)
print("TASK 3: NEW THREAD ID ")
print("=" * 70)

# Ask the agent about the color 
res_mem3 = memory_agent.invoke(
    {"messages": [HumanMessage(content="What is my favorite color?")]},
    config=config_thread_2
)

print("Thread 2 Reply:", res_mem3["messages"][-1].content)

TASK 3: NEW THREAD ID 
Thread 2 Reply: I still don't know your favorite color! Since we haven't discussed it yet in this conversation, I have no way of knowing.

Feel free to share it with me if you'd like! 🎨
